#Init

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

In [0]:
RENAME_MAP = {
    "code" : "product_id",
    "url" : "product_page_url",
    "creator" : "entry_creator",
    "created_datetime" : "product_creation_datetime",
    "last_modified_datetime" : "last_user_modification_datetime",
    "last_updated_datetime" : "last_system_update_datetime",
    "packaging" : "packaging_type",
    "packaging_text" : "packaging_details",
    "brands" : "brands_list",
    "manufacturing_places" : "manufacturing_locations",
    "purchase_places" : "purchasing_locations",
    "ingredients_text" : "ingridients_description",
    "product_quantity" : "product_weight",
    "owner" : "brand_owner",
    "unique_scans_n" : "unique_scan_count",
    "completeness" : "product_data_completeness",
    "last_image_datetime" : "last_product_image_datetime",
    "image_url" : "product_image_url",
    "image_nutrition_url" : "image_nutrition_data_url",
    "energy-kj_100g" : "energy_kj_100g",
    "energy-kcal_100g" : "energy_kcal_100g",
    "saturated-fat_100g" : "saturated_fat_100g",
    "vitamin-a_100g": "vitamin_a_100g",
    "vitamin-d_100g": "vitamin_d_100g",
    "vitamin-e_100g": "vitamin_e_100g",
    "vitamin-k_100g" : "vitamin_k_100g",
    "vitamin-c_100g" : "vitamin_c_100g",
    "vitamin-b1_100g" : "vitamin_b1_100g",
    "vitamin-b2_100g" : "vitamin_b2_100g",
    "vitamin-b6_100g" : "vitamin_b6_100g",
    "vitamin-b9_100g" : "vitamin_b9_100g",
    "vitamin-b12_100g" : "vitamin_b12_100g",
    "carbon-footprint_100g" : "carbon_footprint_100g"
}

#Pulling from Bronze

In [0]:
df = spark.table('openfoodfactslakehouse.bronze.products')

#Cleaning data

In [0]:
#cleaning code
window = Window.partitionBy('code').orderBy(F.desc('last_modified_datetime'))

df = df.withColumn('flag', F.row_number() \
                   .over(window)) \
                   .filter("flag = 1") \
                   .drop('flag')

In [0]:
#cleaning url
df = df.withColumn(
    'url',
    F.when(
        ~F.col('url').contains('http') | F.col('url').isNull(),
        "N/A"
    ).otherwise(F.trim(F.col('url')))
)

In [0]:
#cleaning created_datetime, last_modified_datetime, last_updated_datetime and last_image_datetime
cols = [
    "created_datetime",
    "last_modified_datetime",
    "last_updated_datetime",
    "last_image_datetime"
]

for c in cols:
    df = df.withColumn(
        c,
        F.to_timestamp(c)
    )

In [0]:
#cleaning product_name, packaging, packaging_text, brands, manufacturing_places, purchase_places, ingredients_text, allergens, countries
cols = [
    'product_name',
    'packaging',
    'packaging_text',
    'brands',
    'manufacturing_places',
    'purchase_places',
    'ingredients_text',
    'allergens',
    'countries'
]

for c in cols:
    df = df.withColumn(
        c,
        F.regexp_replace(c, r'\b[a-zA-Z]{2}:', "")
    )
    df = df.withColumn(
        c,
        F.regexp_replace(c, r'^(0?[1-9]|1[0-2])\/\d{4}$', "")
    )

cols_to_capitalize_first_letter = [
    'product_name',
    'packaging',
    'packaging_text',
    'brands',
    'ingredients_text',
    'allergens',
]

for c in cols_to_capitalize_first_letter:
    df = df.withColumn(
        c,
        F.trim(
            F.concat(
                F.upper(F.substring(c, 1, 1)),
                F.lower(F.substring(c, 2, 1000))
            )
        )
    )

cols_to_initcap = [
    'manufacturing_places',
    'purchase_places',
    'countries'
]

for c in cols_to_initcap:
    df = df.withColumn(
        c,
        F.initcap(F.col(c))
    )

df = df.withColumn(
    'brands',
    F.when(
        F.col('brands').rlike(r'[!#"%&\^\*\(\)\-\+;:\'\`\$\~\.\[\]/<>@]') |
        F.col('brands').rlike(r'[0-9]$'),
        'N/A'
    ).otherwise(F.col('brands'))
)

country_map = {
    "us": "United States",
    "usa": "United States",
    "gb": "United Kingdom",
    "uk": "United Kingdom",
    "fr": "France",
    "de": "Germany",
    "ge": "Germany",
    "it": "Italy",
    "es": "Spain",
    "esp": "Spain",
    "pt": "Portugal",
    "nl": "Netherlands",
    "be": "Belgium",
    "ch": "Switzerland",
    "at": "Austria",
    "se": "Sweden",
    "no": "Norway",
    "dk": "Denmark",
    "fi": "Finland",
    "pl": "Poland",
    "cz": "Czech Republic",
    "sk": "Slovakia",
    "hu": "Hungary",
    "ro": "Romania",
    "bg": "Bulgaria",
    "ua": "Ukraine",
    "ca": "Canada",
    "mx": "Mexico",
    "br": "Brazil",
    "ar": "Argentina",
    "au": "Australia",
    "nz": "New Zealand",
    "jp": "Japan",
    "cn": "China",
    "in": "India"
}

country_mapper = F.create_map(
    [F.lit(x) for x in sum(country_map.items(), ())]
)

country_cols = ['countries', 'purchase_places', 'manufacturing_places']
for col in country_cols:
    df = df.withColumn(
        col,
        F.when(
            F.lower(F.col(col)).isin(list(country_map.keys())),
            country_mapper[F.col(col)]
        ).otherwise(F.col(col))
    )

In [0]:
#cleaning serving_quantity
df = df.withColumn(
    'serving_quantity',
    F.when(
        (F.col('serving_quantity').try_cast('float') < 0) | (F.col('serving_quantity').isNull()),
        0
    ).otherwise(F.col('serving_quantity').try_cast('float'))
)
df = df.withColumn(
    'serving_quantity',
    F.round(F.col('serving_quantity'), 2)
)

In [0]:
#cleaning nova_group
df = df.withColumn(
    'nova_group',
    F.when(
        ~F.col('nova_group').isin('1', '2', '3', '4') | (F.col('nova_group').isNull()),
        0
    ).otherwise(F.col("nova_group").try_cast('int'))
)

In [0]:
#cleaning pnns_groups_2
categories = [
    "Dairy desserts",
    "One-dish meals",
    "Sandwiches",
    "Biscuits and cakes",
    "Unsweetened beverages",
    "Nuts",
    "Artificially sweetened beverages",
    "Fats",
    "Sweets",
    "Salty and fatty products",
    "Vegetables",
    "Eggs",
    "Sweetened beverages",
    "Dried fruits",
    "Offals",
    "Cereals",
    "Fruit juices",
    "Bread",
    "Waters and flavored waters",
    "Baby foods",
    "Cheese",
    "Dressings and sauces",
    "Ice cream",
    "Appetizers",
    "Meat",
    "Breakfast cereals",
    "Teas and herbal teas and coffees",
    "Fish and seafood",
    "Legumes",
    "Pizza pies and quiches",
    "Fruit nectars",
    "Pastries",
    "Plant-based milk substitutes",
    "Fruits",
    "Chocolate products",
    "Alcoholic beverages",
    "Baby milks",
    "Processed meat",
    "Milk and yogurt",
    "Potatoes",
    "Soups"
]

df = df.withColumn(
    'pnns_groups_2',
    F.when(
        ~F.col('pnns_groups_2').isin(categories) | F.col('pnns_groups_2').isNull(),
        'N/A'
    ).otherwise(F.initcap(F.col('pnns_groups_2')))
)

In [0]:
#cleaning environmental_score_score and _grade
df = df.withColumn(
    'environmental_score_score',
    F.when(
        (F.col('environmental_score_score').try_cast('int') < 0) | 
        (F.col('environmental_score_score').try_cast('int') > 100) | 
        (F.col('environmental_score_score').isNull()),
        0
    ).otherwise(F.col('environmental_score_score').try_cast('int'))
)
df = df.withColumn(
    'environmental_score_grade',
    F.when(
        ~(F.col('environmental_score_grade').isin('a', 'b', 'c', 'd', 'e', 'f')) |
        (F.col("environmental_score_grade").isNull()),
        'N/A'
    ).otherwise(F.upper(F.col('environmental_score_grade')))
)

In [0]:
#cleaning product_quantity
df = df.withColumn(
  'product_quantity',
  F.when(
    (F.col('product_quantity').try_cast('float') < 0) | 
    F.col('product_quantity').isNull(),
    0
  ).otherwise(F.round(F.col('product_quantity').try_cast('float'), 2))
)

In [0]:
#cleaning owner
df = df.withColumn(
    'owner',
    F.regexp_replace('owner', r'org-', '')
)
df = df.withColumn(
    'owner',
    F.regexp_replace('owner', r'-', ' ')
)
df = df.withColumn(
    'owner',
    F.initcap(F.col('owner'))
)

In [0]:
#cleaning unique_scans_n
df = df.withColumn(
    'unique_scans_n',
    F.when(
    (F.col('unique_scans_n').try_cast('int') < 0) | 
    F.col('unique_scans_n').isNull(),
    0
  ).otherwise(F.col('unique_scans_n').try_cast('int'))
)

In [0]:
#cleaning completeness
df = df.withColumn(
    'completeness',
    F.col("completeness").try_cast('float') * 100
)
df = df.withColumn(
    'completeness',
    F.round(F.col('completeness'), 2)
)
df = df.withColumn(
    'completeness',
    F.concat(F.col('completeness').cast('string'), F.lit('%'))
)

In [0]:
#cleaning image_url columns
df = df.withColumn(
    'image_url',
    F.when(
        ~F.col('image_url').contains('http') | F.col('image_url').isNull(),
        "N/A"
    ).otherwise(F.trim(F.col('image_url')))
)
df = df.withColumn(
    'image_ingredients_url',
    F.when(
        ~F.col('image_ingredients_url').contains('http') | 
        F.col('image_ingredients_url').isNull(),
        "N/A"
    ).otherwise(F.trim(F.col('image_ingredients_url')))
)
df = df.withColumn(
    'image_nutrition_url',
    F.when(
        ~F.col('image_nutrition_url').contains('http') | 
        F.col('image_nutrition_url').isNull(),
        "N/A"
    ).otherwise(F.trim(F.col('image_nutrition_url')))
)

In [0]:
#cleaning all the microelements columns
cols = [
    "energy-kj_100g", "energy-kcal_100g",
    "fat_100g", "saturated-fat_100g", "cholesterol_100g",
    "carbohydrates_100g", "sugars_100g", "lactose_100g", "starch_100g",
    "fiber_100g", "proteins_100g", "salt_100g", "sodium_100g",
    "alcohol_100g", "vitamin-a_100g", "vitamin-d_100g", "vitamin-e_100g",
    "vitamin-k_100g", "vitamin-c_100g", "vitamin-b1_100g",
    "vitamin-b2_100g", "vitamin-b6_100g", "vitamin-b9_100g",
    "vitamin-b12_100g", "potassium_100g", "calcium_100g",
    "phosphorus_100g", "iron_100g", "magnesium_100g", "zinc_100g",
    "copper_100g", "iodine_100g", "caffeine_100g",
    "carbon-footprint_100g"
]

for c in cols:
    df = df.withColumn(
        c,
        F.when(
            (F.col(c).try_cast('float') < 0) |
            F.col(c).isNull(),
            0
        ).otherwise(F.col(c).try_cast('float'))
    )
    df = df.withColumn(
        c,
        F.round(F.col(c), 2)
    )

In [0]:
#data integrity for energy-kcal_100g
df = df.withColumn(
    'energy-kcal_100g',
    F.col("energy-kj_100g") / 4.184
)
df = df.withColumn(
    'energy-kcal_100g',
    F.round(F.col('energy-kcal_100g'), 2)
)

In [0]:
#data integrity for the fat data columns
df = df.withColumn(
    'saturated-fat_100g',
    F.when(
        F.col('saturated-fat_100g') > F.col("fat_100g"),
        0
    ).otherwise(F.col("saturated-fat_100g"))
)

#Filling null values for remaining cols

In [0]:
cols = [
    "code", "creator", "product_name", "packaging", "packaging_text",
    "brands", "manufacturing_places", "purchase_places", "countries",
    "ingredients_text", "allergens", "owner",
    "completeness"
]

df = df.replace('unknown', 'N/A', subset=cols)
df = df.fillna('N/A', subset=cols)

#Renaming the columns

In [0]:
for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

#Writing to Silver layer

In [0]:
(
    df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('openfoodfactslakehouse.silver.products')
)